## Qwen2.5-VL-7B — Train 4,000개 학습과 첫 제출

이 노트북은 smoke LoRA adapter 하나를 이어서 학습합니다. 고정 Train 7,628개 중 seed 42로 고유 ID 4,000개를 선택하고, 각 ID는 이미지 순서를 결정적으로 한 번 재배열한 증강 뷰로 학습합니다. 약 1 epoch, 500 optimizer step으로 제한해 Validation과 Test 추론을 포함한 전체 과정을 4시간 안에 마치는 것이 목표입니다.

학습 후 Train에서 분리한 Validation 50개로 Exact Match를 먼저 확인합니다. 마지막 Test 셀은 추론과 제출 파일 작성만 수행하며 Test 분포 분석은 하지 않습니다. 최종 산출물은 단일 Qwen base model과 단일 LoRA adapter입니다.

### 1. A100과 Google Drive 연결

In [ ]:
import os
import sys
from pathlib import Path
import torch
from google.colab import drive

assert torch.cuda.is_available(), 'GPU runtime이 아닙니다.'
assert torch.cuda.is_bf16_supported(), 'BF16 지원 GPU가 필요합니다.'
drive.mount('/content/drive')
PROJECT_DIR = Path('/content/drive/MyDrive/SNU')
SMOKE_ADAPTER = (
    PROJECT_DIR / 'outputs' / 'qwen25_vl_7b_lora' /
    'smoke_seed42_500x2_100steps_v1' / 'final_adapter'
)
required = [
    PROJECT_DIR / 'data' / 'train.csv',
    PROJECT_DIR / 'data' / 'train',
    PROJECT_DIR / 'data' / 'test.csv',
    PROJECT_DIR / 'data' / 'test',
    PROJECT_DIR / 'data' / 'sample_submission.csv',
    PROJECT_DIR / 'outputs' / 'splits' / 'train_ids.csv',
    PROJECT_DIR / 'outputs' / 'splits' / 'validation_ids.csv',
    PROJECT_DIR / 'scripts' / 'train_qwen_lora.py',
    PROJECT_DIR / 'scripts' / 'evaluate_qwen_baseline.py',
    PROJECT_DIR / 'scripts' / 'predict_qwen_test.py',
    PROJECT_DIR / 'src' / 'qwen_training.py',
    PROJECT_DIR / 'src' / 'submission.py',
    SMOKE_ADAPTER / 'adapter_model.safetensors',
]
for path in required:
    assert path.exists(), f'필수 경로를 찾지 못했습니다: {path}'
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
print(f'PyTorch/CUDA: {torch.__version__}/{torch.version.cuda}')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
print(f'Initial adapter: {SMOKE_ADAPTER}')

### 2. 패키지 설치와 호환되지 않는 선택 패키지 제거

Colab CUDA PyTorch는 교체하지 않습니다. 이번 BF16 LoRA에서 사용하지 않는 구버전 torchao는 PEFT와 충돌하므로 제거합니다.

In [ ]:
%pip install -q transformers==5.13.0 accelerate==1.14.0 qwen-vl-utils==0.0.14 peft==0.19.1
%pip uninstall -y torchao

### 3. 전체 로컬 테스트

모델을 받기 전에 증강 Answer 변환, completion-only loss, split, Test 구조, submission 행 순서를 검사합니다.

In [ ]:
import subprocess

tests_dir = PROJECT_DIR / 'tests'
test_patterns = ['test_qwen*.py', 'test_submission.py']
for pattern in test_patterns:
    result = subprocess.run(
        [
            sys.executable, '-m', 'unittest', 'discover',
            '-s', str(tests_dir), '-p', pattern, '-v',
        ],
        cwd=PROJECT_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    print(result.stdout)
    if result.returncode != 0:
        raise RuntimeError(
            f'{pattern} 테스트 실패 (종료 코드 {result.returncode})'
        )

### 4. 고유 Train 4,000개 학습 범위 사전 확인

선택된 4,000개 source ID가 정확히 한 번 포함되고, 증강 후에도 Answer가 1~4의 순열이며 24개 label이 모두 남아 있는지 확인합니다. Validation은 증강하지 않습니다.

In [ ]:
from src.qwen_training import (
    build_augmented_only_examples,
    choose_training_samples,
    load_labeled_split_samples,
)

all_train = load_labeled_split_samples(
    data_dir=PROJECT_DIR / 'data',
    split_csv=PROJECT_DIR / 'outputs' / 'splits' / 'train_ids.csv',
    expected_split='train',
)
all_validation = load_labeled_split_samples(
    data_dir=PROJECT_DIR / 'data',
    split_csv=PROJECT_DIR / 'outputs' / 'splits' / 'validation_ids.csv',
    expected_split='validation',
)
selected_train = choose_training_samples(all_train, limit=4_000, seed=42)
train_examples = build_augmented_only_examples(selected_train, seed=42)
assert len(all_train) == 7_628
assert len(selected_train) == 4_000
assert len(train_examples) == 4_000
assert len({row.source_sample_id for row in train_examples}) == 4_000
assert all(tuple(sorted(row.sample.answer)) == (1, 2, 3, 4) for row in train_examples)
from collections import Counter
label_counts = Counter(row.sample.answer for row in train_examples)
assert len(label_counts) == 24
print(f'사용 가능한 Train source IDs: {len(all_train):,}')
print(f'선택한 고유 Train: {len(selected_train):,}')
print(f'학습 예시: {len(train_examples):,} (ID당 증강 뷰 1개)')
print(f'Validation 보존: {len(all_validation):,}')
print(f'증강 후 label 종류: {len(label_counts)}')
print(f'label별 샘플 수 범위: {min(label_counts.values())}~{max(label_counts.values())}')
print(f'예상 optimizer steps/epoch: {(len(train_examples) + 7) // 8:,}')

### 5. 고유 Train 4,000개 이어 학습

smoke adapter에서 이어서 단일 adapter 하나만 업데이트합니다. 선택한 4,000개를 1 epoch 학습하고, Validation loss는 원본 100개로 250 step마다 확인합니다. 실제 batch 1과 gradient accumulation 8이므로 정확히 500 optimizer step입니다. 중간 checkpoint는 250 step에 저장합니다.

In [ ]:
MODEL_NAME = 'Qwen/Qwen2.5-VL-7B-Instruct'
CACHE_DIR = Path('/content/huggingface/hub')
RUN_NAME = 'train4000_augmented_once_1epoch_seed42_v1'
TRAIN_RUN_DIR = PROJECT_DIR / 'outputs' / 'qwen25_vl_7b_lora' / RUN_NAME
assert not (TRAIN_RUN_DIR / 'final_adapter').exists(), (
    f'기존 최종 adapter가 있습니다. RUN_NAME을 바꾸세요: {TRAIN_RUN_DIR}'
)

train_command = [
    sys.executable, '-u', '-m', 'scripts.train_qwen_lora',
    '--data-dir', str(PROJECT_DIR / 'data'),
    '--train-split-csv', str(PROJECT_DIR / 'outputs' / 'splits' / 'train_ids.csv'),
    '--validation-split-csv', str(PROJECT_DIR / 'outputs' / 'splits' / 'validation_ids.csv'),
    '--output-dir', str(TRAIN_RUN_DIR),
    '--model-name', MODEL_NAME,
    '--initial-adapter-path', str(SMOKE_ADAPTER),
    '--cache-dir', str(CACHE_DIR),
    '--train-limit', '4000',
    '--validation-limit', '100',
    '--augmentation-copies', '1',
    '--augmented-only',
    '--max-steps', '-1',
    '--epochs', '1',
    '--batch-size', '1',
    '--eval-batch-size', '1',
    '--gradient-accumulation-steps', '8',
    '--learning-rate', '1e-5',
    '--logging-steps', '25',
    '--eval-steps', '250',
    '--save-steps', '250',
    '--save-total-limit', '2',
    '--seed', '42',
]

def run_streaming(command, *, cwd):
    process = subprocess.Popen(
        command, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    recent_lines = []
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='', flush=True)
        recent_lines.append(line.rstrip())
        recent_lines = recent_lines[-60:]
    code = process.wait()
    if code != 0:
        raise RuntimeError(
            f'명령 실패 (종료 코드 {code}). 마지막 로그:\n' + '\n'.join(recent_lines)
        )

run_streaming(train_command, cwd=PROJECT_DIR)

### 6. 학습 완료와 loss 확인

In [ ]:
import json
import pandas as pd
from IPython.display import display

FINAL_ADAPTER = TRAIN_RUN_DIR / 'final_adapter'
assert (FINAL_ADAPTER / 'adapter_model.safetensors').is_file()
training_summary = json.loads((TRAIN_RUN_DIR / 'training_summary.json').read_text(encoding='utf-8'))
run_config = json.loads((TRAIN_RUN_DIR / 'run_config.json').read_text(encoding='utf-8'))
print(json.dumps(training_summary, ensure_ascii=False, indent=2))
print(json.dumps(run_config['split_counts'], ensure_ascii=False, indent=2))
state_path = TRAIN_RUN_DIR / 'checkpoints' / 'trainer_state.json'
if state_path.exists():
    state = json.loads(state_path.read_text(encoding='utf-8'))
    display(pd.DataFrame(state.get('log_history', [])))

### 7. 최종 adapter의 Validation 50개 Exact Match

Test를 열기 전에 smoke와 동일한 Validation 50개로 학습 효과와 파싱을 확인합니다.

In [ ]:
VAL_OUTPUT = TRAIN_RUN_DIR / 'generation_evaluation'
val_command = [
    sys.executable, '-u', '-m', 'scripts.evaluate_qwen_baseline',
    '--data-dir', str(PROJECT_DIR / 'data'),
    '--validation-split-csv', str(PROJECT_DIR / 'outputs' / 'splits' / 'validation_ids.csv'),
    '--output-dir', str(VAL_OUTPUT),
    '--model-name', MODEL_NAME,
    '--adapter-path', str(FINAL_ADAPTER),
    '--prompt-version', 'v2_direct_answer',
    '--cache-dir', str(CACHE_DIR),
    '--device', 'cuda', '--dtype', 'bfloat16',
    '--limit', '50', '--seed', '42', '--max-new-tokens', '16',
]
run_streaming(val_command, cwd=PROJECT_DIR)

### 8. smoke 대비 Validation 성능 확인

공식 판단값은 Exact Match입니다. 이 결과를 확인한 뒤에만 다음 Test 셀을 실행하세요.

In [ ]:
full_val_path = VAL_OUTPUT / 'validation_50' / 'summary.json'
smoke_val_path = (
    PROJECT_DIR / 'outputs' / 'qwen25_vl_7b_lora' /
    'smoke_seed42_500x2_100steps_v1' / 'generation_evaluation' /
    'validation_50' / 'summary.json'
)
full_val = json.loads(full_val_path.read_text(encoding='utf-8'))
smoke_val = json.loads(smoke_val_path.read_text(encoding='utf-8'))
comparison = pd.DataFrame([
    {
        'run': 'smoke 500',
        'exact_match': smoke_val['metrics']['exact_match_accuracy'],
        'position_accuracy': smoke_val['metrics']['position_accuracy'],
        'parse_success': smoke_val['parsing']['success_rate'],
        'unique_answers': smoke_val['predictions']['unique_answer_count'],
        'most_common_fraction': smoke_val['predictions']['most_common_fraction'],
    },
    {
        'run': 'Train 4,000',
        'exact_match': full_val['metrics']['exact_match_accuracy'],
        'position_accuracy': full_val['metrics']['position_accuracy'],
        'parse_success': full_val['parsing']['success_rate'],
        'unique_answers': full_val['predictions']['unique_answer_count'],
        'most_common_fraction': full_val['predictions']['most_common_fraction'],
    },
])
display(comparison)
assert full_val['parsing']['success_rate'] == 1.0, 'Validation 파싱 실패가 있습니다.'
if full_val['metrics']['exact_match_accuracy'] < smoke_val['metrics']['exact_match_accuracy']:
    print('경고: Exact Match가 smoke보다 낮습니다. Test 실행 전 결과를 다시 검토하세요.')
else:
    print('Validation 점검 통과: Test 추론 셀을 실행할 수 있습니다.')

### 9. Test 819개 추론과 submission.csv 생성

이 셀은 Test 정답 분포를 분석하지 않습니다. 단일 최종 adapter로 순서대로 추론하고, 819개 출력이 모두 파싱된 경우에만 `submission.csv`를 만듭니다.

In [ ]:
TEST_OUTPUT = TRAIN_RUN_DIR / 'test_submission'
test_command = [
    sys.executable, '-u', '-m', 'scripts.predict_qwen_test',
    '--data-dir', str(PROJECT_DIR / 'data'),
    '--output-dir', str(TEST_OUTPUT),
    '--model-name', MODEL_NAME,
    '--adapter-path', str(FINAL_ADAPTER),
    '--cache-dir', str(CACHE_DIR),
    '--device', 'cuda', '--dtype', 'bfloat16',
    '--max-new-tokens', '16',
]
run_streaming(test_command, cwd=PROJECT_DIR)

### 10. 제출 직전 구조 검증

Test 내용이나 예측 분포를 분석하지 않고 파일명, 열, 행 수, ID 순서, Answer 순열 유효성만 다시 검사합니다.

In [ ]:
import csv
from src.submission import validate_submission_file

submission_path = TEST_OUTPUT / 'submission.csv'
summary_path = TEST_OUTPUT / 'inference_summary.json'
with (PROJECT_DIR / 'data' / 'sample_submission.csv').open(
    newline='', encoding='utf-8-sig'
) as handle:
    template_ids = [row['Id'] for row in csv.DictReader(handle)]
validated_rows = validate_submission_file(submission_path, template_ids)
inference_summary = json.loads(summary_path.read_text(encoding='utf-8'))
assert validated_rows == 819
assert inference_summary['parse_failure_count'] == 0
print(f'제출 파일: {submission_path}')
print(f'검증된 행: {validated_rows:,}')
print(f'열: Id, Answer')
print('파싱 실패: 0')
print('이 submission.csv 파일을 대회 사이트에 제출하세요.')